In [ ]:
!pip install -q transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.5 MB/s eta 0:00:00


In [ ]:
!pip install -U bitsandbytes>=0.46.1

In [ ]:
# --------  IMPORTS --------
import os
import re
import json
import torch
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig


# --------  NETTOYAGE MÉMOIRE PRÉVENTIF --------
# Libération explicite de la mémoire GPU avant de commencer
gc.collect()
torch.cuda.empty_cache()

# --------  CONFIGURATIONS ET CHEMINS --------
CAMINHO_JSON = "/content/project-7-at-2026-05-06-12-05-670948f7.json"
NOME_RELATORIO = "relatorio_final_llama3.txt"

# -------- CHARGEMENT ET PRÉPARATION DES DONNÉES --------
with open(CAMINHO_JSON, "r", encoding="utf-8") as f:
    dados = json.load(f)

def extrair(exemplo):
    """Extrait les entités annotées depuis le format JSON."""
    out = []
    for r in exemplo.get('annotations', [{}])[0].get('result', []):
        if 'value' in r and 'text' in r['value']:
            out.append({
                "text": r['value']['text'],
                "label": r['value']['labels'][0]
            })
    return out
# Construction du dataset propre : on transforme le JSON brut en une liste
# d'objets Python (dictionnaires) plus facile à manipuler.
dataset = [
    {"id": d.get("id", i), "texto": d["data"]["text"], "labels": extrair(d)}
    for i, d in enumerate(dados)]

# Séparation du jeu de données pour le "Few-Shot Learning" :
# 2 exemples servent à guider le modèle, le reste est utilisé pour l'évaluation.
exemplos_treino = dataset[:2]
textos_teste = dataset[2:]

# -------- SPLIT EN CHUNKS --------
def quebrar_em_blocos(texto, tamanho=3):
    """
    Découpe un texte long en blocs plus petits (chunks).
    Ceci permet de respecter la limite de tokens du modèle et d'améliorer
    la précision d'extraction sur de longs documents.
    """
    # Utilisation d'une expression régulière (regex) pour capturer les
    # signes de ponctuation finale (. ! ?) tout en conservant les espaces.
    frases = re.split(r'(?<=[.!?])\s+', texto)
    blocos = []
    for i in range(0, len(frases), tamanho):
        bloco = " ".join(frases[i:i+tamanho])
        if len(bloco) > 5:
            blocos.append(bloco)
    return blocos

# -------- PROMPT --------
def construire_prompt(exemplos, texto_alvo):
    prompt = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>
Tu es un extracteur d'entités temporelles.

RÈGLES STRICTES:
- Réponds UNIQUEMENT avec des lignes au format: texte exact | LABEL
- Ne traduis JAMAIS le texte
- N’ajoute AUCUNE explication
- Chaque extraction doit être une sous-chaîne EXACTE du texte
- N'invente JAMAIS d'entité
- Réponds uniquement en français
- Si aucune entité: réponds avec une ligne vide

LABELS:
-"ALTE": "Localise une situation par rapport à un événement ou une action. Exemples : 'En même temps que l’Euro de foot', 'Après le discours du pape', 'Lorsque nous arriverons au châlet', 'face au choix d’exposer sa vie'.",
-"ALTC": "Ancrage temporel direct sur un calendrier ou adverbes de temps réguliers. Exemples : 'Aujourd’hui', 'Ces dernières semaines', 'Toujours', 'Souvent', 'Déjà', 'Et puis', '(1939-1945)', 'En direct', 'Dans un premier temps', 'Peu à peu'.",
-"NONALT": "Expression temporelle utilisée comme sujet ou complément d'un nom (n'a pas la fonction de modifieur). Exemples : 'Le 11 juin', 'Tous les autres mois de l’année', 'Les orphelins du 13 Novembre'.",
--"ALTS": "Localisation spatiale stricte (ex: dans cette attaque, à Paris)."
- "héméronyme": Uniquement une date avec tiret ou # par exemple: "13-Novembre" .

FORMAT:
texte | LABEL<|eot_id|>
"""
# Nous ajoutons les exemples d'entraînement au prompt pour que le modèle
    # "comprenne" le format attendu (Texte | Label) avant de traiter la cible.
    for ex in exemplos:
      # Format spécifique pour Llama 3 (ChatML/Instruct)
        prompt += '<|start_header_id|>user<|end_header_id|>\n'
        prompt += f'Texte: "{ex["texto"]}"<|eot_id|>\n'
        prompt += '<|start_header_id|>assistant<|end_header_id|>\n'

        for an in ex["labels"]:
          # Ajout des annotations réelles pour cet exemple
            prompt += f'{an["text"]} | {an["label"]}\n'

        prompt += '<|eot_id|>\n'
# Enfin, on ajoute le texte réel que le modèle doit analyser
    prompt += '<|start_header_id|>user<|end_header_id|>\n'
    prompt += f'Texte: "{texto_alvo}"<|eot_id|>\n'
    prompt += '<|start_header_id|>assistant<|end_header_id|>\n'

    return prompt

# -------- CHARGEMENT ET QUANTIFICATION DU MODÈLE --------
print(" Chargement du modèle...")

model_id = "unsloth/llama-3-8b-Instruct-bnb-4bit"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True)

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto")

model.config.use_cache = False

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer)

# -------- NETTOYAGE --------
def limpar_linha(linha):
    linha = linha.replace("*", "").replace("Label:", "").strip()
    return linha

def limpar_texto(txt):
    return txt.strip().lower()

def clean_punct(txt):
    txt = txt.lower()
    txt = txt.replace("(", "").replace(")", "")
    txt = txt.replace(",", "").replace(".", "")
    txt = txt.replace(";", "")
    txt = txt.replace("!", "").replace("?", "")
    txt = txt.replace("’", "'")
    txt = re.sub(r"\s+", " ", txt).strip()
    return txt

def formatar_lista(lista):
    if not lista:
        return "  (aucun)"
    return "\n".join([f"  • {t} -> {l}" for t, l in lista])

# -------- BOUCLE D'INFÉRENCE ET ANALYSE DES PRÉDICTIONS--------
total_tp, total_fp, total_fn = 0, 0, 0
todas_avaliacoes = []

for item in textos_teste:
# Découpage du texte en segments (chunks) pour respecter la limite du LLM
    blocos = quebrar_em_blocos(item["texto"])
    parsed_llm = []
    raw_global = ""

    for bloco in blocos:
# Construction du prompt avec les exemples de référence (Few-shot)
        prompt = construire_prompt(exemplos_treino, bloco)
# Génération de texte sans échantillonnage (température 0 )
        with torch.no_grad():
            saida = generator(
                prompt,
                max_new_tokens=80,
                temperature=0.0,
                do_sample=False,
                repetition_penalty=1.1,
                eos_token_id=tokenizer.eos_token_id
            )
# Extraction de la réponse du modèle
        raw = saida[0]["generated_text"].split("<|start_header_id|>assistant<|end_header_id|>")[-1]
        raw_global += raw + "\n"
# Parsing ligne par ligne de la réponse du modèle
        for linha in raw.split("\n"):
            linha = limpar_linha(linha)
# Si la ligne contient notre séparateur '|', on extrait texte et labe
            if "|" in linha:
                partes = linha.split("|")
                if len(partes) >= 2:
                    texto = partes[0].strip()
                    label = partes[1].strip()
# Validation : on ne garde que les labels attendus et non vides
                    if label in ["ALTE", "ALTC", "NONALT"] and texto:
                        parsed_llm.append((limpar_texto(texto), label))
# Libération immédiate de la mémoire après chaque bloc pour éviter la saturation
        del saida
        torch.cuda.empty_cache()

    #  enlever doublons
    parsed_llm = list(set(parsed_llm))

    # -------- ÉVALUATION --------
    gold = [(clean_punct(g["text"]), g["label"]) for g in item["labels"]]
    pred = [(clean_punct(p[0]), p[1]) for p in parsed_llm]

    tp, fp = 0, 0
    gold_copy = gold[:]

    matched = []
    fp_list = []

    for p in pred:
        if p in gold_copy:
            tp += 1
            matched.append(p)
            gold_copy.remove(p)
        else:
            fp += 1
            fp_list.append(p)

    fn_list = gold_copy
    fn = len(fn_list)

    total_tp += tp
    total_fp += fp
    total_fn += fn

    todas_avaliacoes.append({
        "id": item["id"],
        "texto": item["texto"],
        "gold": gold,
        "pred": pred,
        "tp": matched,
        "fp": fp_list,
        "fn": fn_list,
        "raw": raw_global
    })

# -------- MÉTRIQUES --------
precision = total_tp / (total_tp + total_fp) if total_tp + total_fp else 0
recall = total_tp / (total_tp + total_fn) if total_tp + total_fn else 0
f1 = (2 * precision * recall) / (precision + recall) if precision + recall else 0

print("\n RÉSULTATS:")
print(f"Precision: {precision:.2%}")
print(f"Recall: {recall:.2%}")
print(f"F1-score: {f1:.2%}")

# -------- RAPPORT --------
with open(NOME_RELATORIO, "w", encoding="utf-8") as f:

    f.write(f"Precision: {precision:.2%}\n")
    f.write(f"Recall: {recall:.2%}\n")
    f.write(f"F1-score: {f1:.2%}\n")
    f.write("="*80 + "\n\n")

    for doc in todas_avaliacoes:

        f.write(f" DOCUMENT ID: {doc['id']}\n")
        f.write(f"TEXTE (tronqué): {doc['texto'][:200]}...\n\n")
        f.write("=== GOLD ===\n")
        f.write(formatar_lista(doc["gold"]) + "\n\n")
        f.write("=== PREDICTION ===\n")
        f.write(formatar_lista(doc["pred"]) + "\n\n")
        f.write("===  TRUE POSITIVES ===\n")
        f.write(formatar_lista(doc["tp"]) + "\n\n")
        f.write("===  FALSE POSITIVES ===\n")
        f.write(formatar_lista(doc["fp"]) + "\n\n")
        f.write("===  FALSE NEGATIVES ===\n")
        f.write(formatar_lista(doc["fn"]) + "\n\n")
        f.write("===  RAW OUTPUT ===\n")
        f.write(doc["raw"] + "\n\n")
        f.write("-" * 80 + "\n\n")

print(f"\n Rapport sauvegardé dans : {NOME_RELATORIO}")